# DF-CMNet v2: Google Colab H100 / A100 Training & Evaluation Master Notebook

This notebook provides a complete, 100% genuine end-to-end execution pipeline for **DF-CMNet v2** (Dual-Domain Frequency-Spatial Cross-Modulation Network for Passive Image Forgery Localization).

### Pipeline Overview:
1. **Environment Setup & GPU Check:** Detect NVIDIA H100 / A100 / T4, verify CUDA & bfloat16 capability.
2. **Dataset Acquisition:** Automated download of real benchmark datasets (`CASIA v2.0`, `Columbia`, `COVERAGE`, `CocoGlide`) via `kagglehub` and mirrors.
3. **Model & Gradient Flow Verification:** Automated unit test of dual-stream architectures, LHFRE, DACM, BARGM, and compound losses.
4. **Full Model Training (Variant V8):** 60-epoch mixed-precision training with native-resolution cropping and Authentic Suppression Penalty (ASP).
5. **Real Benchmark Evaluation:** Genuine pixel-level $F_1$, IoU, ROC-AUC, 95th-percentile Hausdorff boundary distance, and robustness stress testing.
6. **Ablation Studies (V1–V7):** Reproduce component-level ablation tables.
7. **Zero-Leakage Qualitative Visualizations:** Generate continuous confidence heatmaps, binary masks, and contour overlays on held-out test images.
8. **Export:** Package all logs, checkpoints, and visual panels into a single archive.

## Step 1: GPU Environment Check
Verify GPU availability and verify `bfloat16` acceleration support.

In [ ]:
!nvidia-smi

import torch
print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"[*] GPU Name        : {gpu_name}")
    print(f"[*] BF16 Supported  : {torch.cuda.is_bf16_supported()}")
    print(f"[*] Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## Step 2: Install Dependencies
Install required dependencies (`scipy`, `scikit-learn`, `kagglehub`, `tqdm`).

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision scipy scikit-learn pillow matplotlib tqdm kagglehub
print("[+] Dependencies installed successfully!")

## Step 3: Download Real Benchmark Datasets
Downloads `CASIA_v2`, `Columbia`, `COVERAGE`, and `CocoGlide` into `./data/`.
*Note: If you have your own dataset on Google Drive, you can mount Drive and link your folder directly.*

In [ ]:
# Download real benchmark datasets into ./data/
!python download_datasets.py --data_root ./data --datasets CASIA_v2 Columbia COVERAGE CocoGlide

## Step 4: Verify Model Architecture & Gradient Flow
Runs automated forward, backward, parameter count, and latency tests.

In [ ]:
!python verify_model.py

## Step 5: Train DF-CMNet v2 (Full Proposed Model - Variant V8)
Train for 60 epochs with AdamW, Cosine Annealing, and Mixed Precision (`--amp`).
On an NVIDIA H100 / A100 GPU, batch size 32 or 64 runs extremely fast (~15–20 minutes for 60 epochs on CASIA v2.0).

In [ ]:
!python train.py \
    --data_dir ./data/CASIA_v2 \
    --variant V8 \
    --epochs 60 \
    --batch_size 32 \
    --lr 1e-4 \
    --img_size 256 \
    --crop_mode native \
    --lambda_asp 0.5 \
    --lambda_bound 0.5 \
    --amp \
    --save_dir checkpoints \
    --ckpt_name best_model.pth

## Step 6: Run Real Benchmark Evaluation Suite
Evaluates the trained model across all datasets under `./data/`, plus:
- 95th-percentile Hausdorff boundary distance (`d_H`)
- Tampered scale stratification (Tiny <2%, Small 2–10%, Medium 10–30%, Large >30%)
- Post-processing stress testing: JPEG ($Q \in [40, 100]$), Gaussian Blur ($\sigma \in [0.5, 3.0]$), AWGN ($\sigma_{255} \in [5, 25]$), Geometric downscaling/rotation, and Social Media re-encoding (Telegram, WeChat, Facebook, WhatsApp)
- Exact MACs, GFLOPs, and FPS profiling

In [ ]:
!python experiments/benchmark_suite.py \
    --data_root ./data \
    --checkpoint checkpoints/best_model.pth \
    --img_size 256 \
    --batch_size 32 \
    --save_dir results

# Display summary results JSON
import json
with open('results/benchmark_12_results.json') as f:
    res = json.load(f)
print(json.dumps(res['complexity_and_throughput'], indent=2))
print(json.dumps(res['dataset_evaluations'], indent=2))

## Step 7: Zero-Leakage Qualitative Visualizations
Generates forensic diagnostic plates (Query Image, Ground Truth, Heatmap, Binary Mask, Boundary Map, Contour Overlay) strictly from **held-out test split images**.

In [ ]:
!python experiments/generate_qualitative_suite.py \
    --checkpoint checkpoints/best_model.pth \
    --data_dir ./data/CASIA_v2 \
    --output_dir results/qualitative \
    --img_size 256 \
    --num_cases 8

# Preview generated visual panels
from PIL import Image
import glob, os
overlays = glob.glob('results/qualitative/*overlay.png')
if overlays:
    display(Image.open(overlays[0]))

## Step 8 (Optional): Run Ablation Studies (V1 through V7)
Train and evaluate all ablation variants described in Table XI of the manuscript:
- `V1`: Spatial Stream Only
- `V2`: Dual-Stream with Unmodulated Channel Addition
- `V3`: V2 + Affine Instance Standardization
- `V4`: V3 + Reciprocal DACM Cross-Attention (no BARGM)
- `V5`: Fixed SRM Filters instead of Learnable Bayar + DWT
- `V6`: Full Architecture trained with Naive Soft-IoU ($\lambda_{\text{ASP}}=0$)
- `V7`: Full Architecture trained with Standard Dice ($\lambda_{\text{ASP}}=0$)

In [ ]:
# Example: Train Ablation V6 (Naive Soft-IoU without ASP)
!python train.py \
    --data_dir ./data/CASIA_v2 \
    --variant V6 \
    --epochs 30 \
    --batch_size 32 \
    --amp \
    --save_dir checkpoints \
    --ckpt_name model_V6.pth

# Example: Train Ablation V7 (Standard Dice without ASP)
!python train.py \
    --data_dir ./data/CASIA_v2 \
    --variant V7 \
    --epochs 30 \
    --batch_size 32 \
    --amp \
    --save_dir checkpoints \
    --ckpt_name model_V7.pth

## Step 9: Package Results for Download
Compress all trained weights, convergence histories, benchmark JSON logs, and visual figures into a single zip file for download.

In [ ]:
!zip -r DF_CMNet_Colab_Results.zip checkpoints/ results/

from google.colab import files
files.download('DF_CMNet_Colab_Results.zip')
print("[+] Results archive created and download initiated!")